# 08 — Thesis analysis: calibration, skill maps, blending, case studies

Runs locally on CPU from the saved predictions of notebook 07 (`outputs/kaggle_v3/stageB/preds`). No model is run.
Fitted quantities (Platt calibration, blend weight, alarm threshold) use the validation years 2011-2014 only; everything shown is on the test years 2015-2018.
Run from the repository root (or set `REPO`).

In [ ]:
import os, sys, subprocess
import pandas as pd
from IPython.display import Image, display
REPO = os.path.abspath(".")
PREDS = "outputs/kaggle_v3/stageB/preds"
DATA = "regional_climax_dataset_6x6_corrected"
OUT, FIG = "docs/mtech/results/analysis", "docs/mtech/figures/analysis"
subprocess.run([sys.executable, "scripts/mtech/make_analysis.py", "--preds_dir", PREDS, "--data_dir", DATA,
                "--out_dir", OUT, "--fig_dir", FIG], cwd=REPO, check=True)

## Overall test scores (PR-AUC primary; Brier skill vs climatology; ECE = calibration error)

In [ ]:
ov = pd.read_csv(f"{OUT}/overall_test.csv")
display(ov.pivot_table(index="source", columns="hazard", values=["pr_auc", "brier_skill_vs_clim", "ece"]).round(3))

## Reliability: raw frozen-head probabilities are over-confident (class-weighted training); Platt scaling fixes it without changing PR-AUC

In [ ]:
display(Image(f"{FIG}/reliability.png"))

## Where ClimaX beats logistic regression

In [ ]:
display(Image(f"{FIG}/skill_maps.png"))

## ClimaX + logistic blend (weight chosen on validation)

In [ ]:
display(pd.read_csv(f"{OUT}/blend_bootstrap.csv")[["a", "b", "hazard", "pr_auc_diff", "ci_low", "ci_high"]].round(4))

## Case studies: forecasts issued 3 days ahead

In [ ]:
display(pd.read_csv(f"{OUT}/cases_summary.csv").drop(columns=["cells"]).round(3))
for f in sorted(os.listdir(FIG)):
    if f.startswith("case_"):
        display(Image(f"{FIG}/{f}"))